# Session X: Lab — Building an Interactive Personal Budget Manager

**Goal:** Combine everything from Sessions I–IX into one complete application.

---

## Exercise 1: Define the Transaction Model

Create a `Transaction` class with:
- `INCOME_CATEGORIES` and `EXPENSE_CATEGORIES` class attributes
- `__init__`: amount, category, description, is_income (default False), auto-set date. Raise `ValueError` if the amount is not a positive, finite number (use `math.isfinite()`) or if the category is not in the list for that transaction type
- `to_dict()` and `@classmethod from_dict()` for serialization
- `__repr__` for readable output
- Type hints on all methods

### How to approach this code

A Transaction class is a blueprint for one record. The two category lists are shared class data, while `__init__` checks the amount and category before storing each object's values; `to_dict()` and `from_dict()` provide the two directions between objects and JSON-friendly dictionaries. `__repr__()` makes objects readable in debugging. The test checks that values survive a serialization round trip and that invalid data is rejected.

In [3]:
from __future__ import annotations
import math
from datetime import datetime


class Transaction:
    INCOME_CATEGORIES = ["Salary", "Freelance", "Gift", "Other income"]
    EXPENSE_CATEGORIES = ["Food", "Transport", "Entertainment", "Housing", "Utilities", "Other"]

    def __init__(self, amount: float, category: str, description: str,
                 is_income: bool = False) -> None:
        # TODO: raise ValueError for a bad amount or category, then store the fields
        if not isinstance(amount, (int, float)) or not math.isfinite(amount) or amount <= 0:
            raise ValueError("Amount must be a positive, finite number")

        if is_income:
            if category not in self.INCOME_CATEGORIES:
                raise ValueError("Invalid income category")
        else:
            if category not in self.EXPENSE_CATEGORIES:
                raise ValueError("Invalid expense category")

        self.amount = amount
        self.category = category
        self.description = description
        self.is_income = is_income
        self.date = datetime.now().strftime("%Y-%m-%d")

    def to_dict(self) -> dict:
        # TODO
        return {
            "amount": self.amount,
            "category": self.category,
            "description": self.description,
            "is_income": self.is_income,
            "date": self.date
        }

    @classmethod
    def from_dict(cls, data: dict) -> Transaction:
        # TODO
        transaction = cls(
            data["amount"],
            data["category"],
            data["description"],
            data["is_income"]
        )
        transaction.date = data["date"]
        return transaction

    def __repr__(self) -> str:
        # TODO
        transaction_type = "Income" if self.is_income else "Expense"
        return f"Transaction({transaction_type}, {self.amount}, {self.category}, {self.description})"


# Test
t1 = Transaction(25.50, "Food", "Lunch")
print(t1)
t2 = Transaction(3000, "Salary", "Monthly", is_income=True)
print(t2)
data = t1.to_dict()
t1_copy = Transaction.from_dict(data)
print(f"Round-trip: {t1.amount == t1_copy.amount}")
try:
    Transaction(-5, "Food", "Refund?")
except ValueError as error:
    print("Rejected:", error)
try:
    Transaction(10, "Salary", "Not an expense")
except ValueError as error:
    print("Rejected:", error)

Transaction(Expense, 25.5, Food, Lunch)
Transaction(Income, 3000, Salary, Monthly)
Round-trip: True
Rejected: Amount must be a positive, finite number
Rejected: Invalid expense category


---
## Exercise 2: Input Validation Functions

Write:
- `get_valid_amount(prompt)` — positive, finite number (reject `nan` and `inf` with `math.isfinite()`), catches ValueError
- `get_valid_category(categories)` — numbered menu, validates int input
- `get_valid_description()` — rejects empty strings

### How to approach this code

Validation first converts input text, then checks whether the converted value follows the app's rules. A retry loop lets users recover from nonnumeric amounts; separate checks reject zero, negative, non-finite (`nan`, `inf`) or out-of-range values. Reusable helpers keep those rules consistent across menu options.

In [6]:
import math


def get_valid_amount(prompt: str) -> float:
    # TODO: while True + try/except; reject non-finite and non-positive values
    while True:
        try:
            amount = float(input(prompt))

            if math.isfinite(amount) and amount > 0:
                return amount
            else:
                print("Amount must be a positive, finite number.")

        except ValueError:
            print("Please enter a valid number.")


def get_valid_category(categories: list[str]) -> str:
    # TODO: numbered menu + validation
    while True:
        for i, category in enumerate(categories, start=1):
            print(f"{i}. {category}")

        try:
            choice = int(input("Choose a category: "))

            if 1 <= choice <= len(categories):
                return categories[choice - 1]
            else:
                print("Invalid choice. Please select a number from the menu.")

        except ValueError:
            print("Please enter a valid number.")


def get_valid_description() -> str:
    # TODO: reject empty
    while True:
        description = input("Enter description: ").strip()

        if description:
            return description
        else:
            print("Description cannot be empty.")


# Uncomment to test:
print(get_valid_amount("Amount: $"))

Amount must be a positive, finite number.
Please enter a valid number.
Amount must be a positive, finite number.
Amount must be a positive, finite number.
343.343


---
## Exercise 3: JSON Persistence

Write `save_transactions(transactions, filepath)` and `load_transactions(filepath)`.
- Save through a temporary file and `os.replace()`, so a failed write never destroys the old file.
- A missing file (`FileNotFoundError`) is a normal first run: return `[]`.
- A file that is not valid JSON (`JSONDecodeError`), or valid JSON that is not a list, is damaged: move it to `filepath + ".bak"`, print a warning and return `[]`.

### How to approach this code

The save and load functions make in-memory transactions persist between program runs. Writing to a temporary file and swapping it in with `os.replace()` keeps the old file intact if the write fails. `FileNotFoundError` means a first run, while a damaged file is set aside as `.bak` so it is never overwritten. The test uses temporary paths to avoid changing real data.

In [7]:
import json
import os
import tempfile


def save_transactions(transactions: list[dict], filepath: str = "data/budget.json") -> None:
    # TODO: write a temporary file, then os.replace() it into place
    folder = os.path.dirname(filepath)

    if folder:
        os.makedirs(folder, exist_ok=True)

    temp_filepath = None

    try:
        with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=folder or ".", delete=False) as temp_file:
            temp_filepath = temp_file.name
            json.dump(transactions, temp_file, indent=4)

        os.replace(temp_filepath, filepath)

    finally:
        if temp_filepath is not None and os.path.exists(temp_filepath):
            os.remove(temp_filepath)


def load_transactions(filepath: str = "data/budget.json") -> list[dict]:
    # TODO: missing file -> []; damaged or wrong shape -> move to .bak, warn, []
    try:
        with open(filepath, "r", encoding="utf-8") as file:
            transactions = json.load(file)

        if not isinstance(transactions, list):
            raise ValueError("JSON file does not contain a list")

        return transactions

    except FileNotFoundError:
        return []

    except (json.JSONDecodeError, ValueError):
        backup_filepath = filepath + ".bak"
        os.replace(filepath, backup_filepath)
        print(f"Warning: Damaged budget file moved to {backup_filepath}")
        return []


# Test
test_data = [{"amount": 50, "category": "Food", "description": "Groceries", "is_income": False, "date": "2026-10-01"}]
save_transactions(test_data, "/tmp/test_budget.json")
loaded = load_transactions("/tmp/test_budget.json")
print(loaded)
print(load_transactions("/tmp/nonexistent.json"))
with open("/tmp/damaged_budget.json", "w", encoding="utf-8") as f:
    f.write('{"amount": 5,')
print(load_transactions("/tmp/damaged_budget.json"))
print(os.path.exists("/tmp/damaged_budget.json.bak"))

[{'amount': 50, 'category': 'Food', 'description': 'Groceries', 'is_income': False, 'date': '2026-10-01'}]
[]
[]
True


---
## Exercise 4: CSV Export

Write `export_to_csv(transactions, filepath)` using csv.DictWriter. Create the folder if it is missing, open the file with `newline=""` and `encoding="utf-8"`, and handle an empty list.

### How to approach this code

CSV export maps each transaction dictionary to a row with a stable set of columns. `writeheader()` labels the fields and `writerows()` writes records; the empty-list check prevents a false success message. Reading the temporary export back verifies the actual output.

In [8]:
import csv
import os


def export_to_csv(transactions: list[dict], filepath: str = "data/export.csv") -> None:
    # TODO
    if not transactions:
        print("No transactions to export.")
        return

    folder = os.path.dirname(filepath)

    if folder:
        os.makedirs(folder, exist_ok=True)

    fieldnames = ["date", "category", "description", "amount", "is_income"]

    with open(filepath, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(transactions)

    print(f"Transactions exported successfully to {filepath}")


# Test
test_data = [{"date":"2026-10-01","category":"Food","description":"Groceries","amount":50.0,"is_income":False}]
export_to_csv(test_data, "/tmp/test_export.csv")
with open("/tmp/test_export.csv", encoding="utf-8") as f:
    print(f.read())
export_to_csv([])

Transactions exported successfully to /tmp/test_export.csv
date,category,description,amount,is_income
2026-10-01,Food,Groceries,50.0,False

No transactions to export.


---
## Exercise 5: Summary Report

Write `generate_report(transactions)` that calculates income/expenses/balance, groups expenses by category and shows percentages. Handle an empty list, and print no "Expenses by Category" heading for an income-only list.

### How to approach this code

The report calculates income, expenses, balance, and totals by category from the same transaction list. Generator expressions select records before summing; a dictionary accumulates category totals. Print the category block only when there is at least one expense: then `expenses > 0`, so the percentage division is safe, and an income-only report has no empty heading.

In [9]:
def generate_report(transactions: list[dict]) -> None:
    # TODO
    if not transactions:
        print("No transactions to report.")
        return

    income = sum(transaction["amount"] for transaction in transactions if transaction["is_income"])
    expenses = sum(transaction["amount"] for transaction in transactions if not transaction["is_income"])
    balance = income - expenses

    print("\n===== Financial Summary =====")
    print(f"Total Income: €{income:.2f}")
    print(f"Total Expenses: €{expenses:.2f}")
    print(f"Balance: €{balance:.2f}")

    expense_categories = {}

    for transaction in transactions:
        if not transaction["is_income"]:
            category = transaction["category"]
            expense_categories[category] = expense_categories.get(category, 0) + transaction["amount"]

    if expenses > 0:
        print("\n===== Expenses by Category =====")

        for category, amount in expense_categories.items():
            percentage = (amount / expenses) * 100
            print(f"{category}: €{amount:.2f} ({percentage:.1f}%)")

test_data = [
    {"amount": 3000, "category": "Salary", "description": "Monthly", "is_income": True},
    {"amount": 800, "category": "Housing", "description": "Rent", "is_income": False},
    {"amount": 150, "category": "Food", "description": "Groceries", "is_income": False},
    {"amount": 75, "category": "Food", "description": "Restaurants", "is_income": False},
    {"amount": 50, "category": "Transport", "description": "Metro", "is_income": False},
]
generate_report(test_data)
generate_report([])
generate_report(test_data[:1])   # income only: no "Expenses by Category" heading


===== Financial Summary =====
Total Income: €3000.00
Total Expenses: €1075.00
Balance: €1925.00

===== Expenses by Category =====
Housing: €800.00 (74.4%)
Food: €225.00 (20.9%)
Transport: €50.00 (4.7%)
No transactions to report.

===== Financial Summary =====
Total Income: €3000.00
Total Expenses: €0.00
Balance: €3000.00


---
## Exercise 6: Main Application Loop

Wire everything together with a menu: Add Income, Add Expense, View All, Report, Export CSV, Save & Quit, Quit.
- At start-up, load the saved records and convert them with `clean_records()`, which skips (and reports) any record that fails validation.
- `ask_transaction(is_income)` asks for the amount, a category from the right list, and a description.
- Report, export and save work with dictionaries: convert with `[t.to_dict() for t in transactions]`.
- Q asks before discarding unsaved changes; any other key prints an error message.

### How to approach this code

The main loop coordinates the application: it displays choices and calls the relevant validation, model, reporting, export, and storage operations. Keep those implementations in their own functions so each menu branch remains easy to read. Saving before exit differs from quitting without saving, so Q should warn when there are unsaved changes.

In [10]:
def clean_records(records: list[dict]) -> list[Transaction]:
    # TODO: from_dict() each record; report and skip the ones that raise an error
    transactions = []

    for record in records:
        try:
            transaction = Transaction.from_dict(record)
            transactions.append(transaction)
        except (ValueError, KeyError, TypeError) as error:
            print(f"Skipping invalid record: {error}")

    return transactions


def ask_transaction(is_income: bool) -> Transaction:
    # TODO: amount, a category from the right list, description
    amount = get_valid_amount("Enter amount: €")

    if is_income:
        categories = Transaction.INCOME_CATEGORIES
    else:
        categories = Transaction.EXPENSE_CATEGORIES

    category = get_valid_category(categories)
    description = get_valid_description()

    return Transaction(amount, category, description, is_income)


def main() -> None:
    transactions = clean_records(load_transactions())  # TODO: load the saved records and clean them
    unsaved = False

    while True:
        print("\n=== PERSONAL BUDGET MANAGER ===")
        print("[1] Add Income  [2] Add Expense  [3] View All")
        print("[4] Report  [5] Export CSV  [6] Save & Quit  [Q] Quit")

        choice = input("Select: ").strip().lower()

        if choice == "1":
            # TODO
            transaction = ask_transaction(True)
            transactions.append(transaction)
            unsaved = True
            print("Income added successfully.")

        elif choice == "2":
            # TODO
            transaction = ask_transaction(False)
            transactions.append(transaction)
            unsaved = True
            print("Expense added successfully.")

        elif choice == "3":
            # TODO
            if not transactions:
                print("No transactions found.")
            else:
                print("\n=== ALL TRANSACTIONS ===")
                for transaction in transactions:
                    print(transaction)

        elif choice == "4":
            # TODO
            generate_report([t.to_dict() for t in transactions])

        elif choice == "5":
            # TODO
            export_to_csv([t.to_dict() for t in transactions])

        elif choice == "6":
            # TODO: save and break
            save_transactions([t.to_dict() for t in transactions])
            print("Transactions saved successfully.")
            print("Goodbye!")
            break

        elif choice == "q":
            # TODO: if there are unsaved changes, ask before quitting
            if unsaved:
                confirm = input("You have unsaved changes. Quit without saving? (y/n): ").strip().lower()

                if confirm == "y":
                    print("Goodbye!")
                    break
                elif confirm == "n":
                    print("Returning to menu.")
                else:
                    print("Invalid input. Returning to menu.")
            else:
                print("Goodbye!")
                break

        else:
            print("Invalid option.")

# Uncomment to run (Ctrl-D or the stop button ends the input):
main()


=== PERSONAL BUDGET MANAGER ===
[1] Add Income  [2] Add Expense  [3] View All
[4] Report  [5] Export CSV  [6] Save & Quit  [Q] Quit
Invalid option.

=== PERSONAL BUDGET MANAGER ===
[1] Add Income  [2] Add Expense  [3] View All
[4] Report  [5] Export CSV  [6] Save & Quit  [Q] Quit
1. Salary
2. Freelance
3. Gift
4. Other income
Income added successfully.

=== PERSONAL BUDGET MANAGER ===
[1] Add Income  [2] Add Expense  [3] View All
[4] Report  [5] Export CSV  [6] Save & Quit  [Q] Quit

=== ALL TRANSACTIONS ===
Transaction(Income, 500.0, Freelance, online work)

=== PERSONAL BUDGET MANAGER ===
[1] Add Income  [2] Add Expense  [3] View All
[4] Report  [5] Export CSV  [6] Save & Quit  [Q] Quit
1. Salary
2. Freelance
3. Gift
4. Other income
Income added successfully.

=== PERSONAL BUDGET MANAGER ===
[1] Add Income  [2] Add Expense  [3] View All
[4] Report  [5] Export CSV  [6] Save & Quit  [Q] Quit
1. Food
2. Transport
3. Entertainment
4. Housing
5. Utilities
6. Other
Please enter a valid nu